# PRJNA566284 16S SPECTRA workflow

This notebook uses a relative-abundance matrix with samples in rows and features in columns. Feature names must match the MPA annotations used by SPECTRA-16S. The released model performs preprocessing internally and returns SPECTRA probabilities for evaluation.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings(
    "ignore",
    message=r"urllib3 .* doesn't match a supported version!",
)

from sklearn.metrics import roc_auc_score

WORK_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    path
    for path in (WORK_DIR, *WORK_DIR.parents)
    if (path / "SPECTRA_GitHub_resource").is_dir()
)
RESOURCE_ROOT = PROJECT_ROOT / "SPECTRA_GitHub_resource"
sys.path.insert(0, str(RESOURCE_ROOT / "scripts"))

from utils import predict_from_abundance_to_phenotype

ABUNDANCE_FILE = WORK_DIR / "4. Name-converted relative abundance.csv"
METADATA_FILE = WORK_DIR / "0. metadata.csv"
MRI_FILE = WORK_DIR / "5. MRI scores.csv"
PROBABILITY_FILE = WORK_DIR / "6. Probability.csv"

MRI_MODEL = RESOURCE_ROOT / "models/extensions/16S/mri_models_all.pkl"
SPECTRA_MODEL = RESOURCE_ROOT / 'models/extensions/16S/spectra_16s_model.pkl'

# 1. Relative-abundance input

In [2]:
abundance = pd.read_csv(ABUNDANCE_FILE, index_col=0, float_precision="round_trip")
abundance.index = abundance.index.astype(str)

display(pd.DataFrame({
    "value": [abundance.shape[0], abundance.shape[1]],
}, index=["samples", "observed mapped features"]))
display(abundance.head())


,value
samples,29
observed mapped features,177


,1502_Clostridium perfringens,Unmatched_taxon_1,Unmatched_taxon_2,40520_Blautia obeum,Unmatched_taxon_3,39486_Dorea formicigenerans,Unmatched_taxon_4,Unmatched_taxon_5,Unmatched_taxon_6,Unmatched_taxon_7,...,Unmatched_taxon_136,Unmatched_taxon_137,Unmatched_taxon_138,Unmatched_taxon_139,Unmatched_taxon_140,Unmatched_taxon_141,Unmatched_taxon_142,Unmatched_taxon_143,Unmatched_taxon_144,Unmatched_taxon_145
sample_id,,,,,,,,,,,,,,,,,,,,,
SRR10142826,0.495694,0.163968,0.000344,0.044092,0.011023,0.038925,0.058905,0.047882,0.004478,0.007923,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR10142827,0.017041,0.150345,0.000647,0.160915,0.089085,0.041846,0.242235,0.059965,0.004745,0.004961,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR10142828,0.031228,0.173684,0.000000,0.164912,0.111579,0.027719,0.106316,0.143333,0.004561,0.010175,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR10142829,0.058242,0.152747,0.000000,0.094505,0.017582,0.023077,0.102198,0.053846,0.001099,0.010989,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR10142830,0.050206,0.097475,0.000000,0.051380,0.017910,0.023194,0.427187,0.048738,0.002936,0.010570,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# 2. SPECTRA prediction

In [3]:
result = predict_from_abundance_to_phenotype(
    Abundance=abundance,
    MRI_model_path=MRI_MODEL,
    SPECTRA_model_path=SPECTRA_MODEL,
)

mri = result["MRI"]
probability = result["probability"]
mri.to_csv(MRI_FILE)
probability.to_csv(PROBABILITY_FILE)

display(mri.head())
display(probability.head())

,control,Intestinal Diseases,Lung Diseases,IBS,Thyroid Diseases,ColorectalLesions
sample_id,,,,,,
SRR10142826,0.514,0.322,0.260,0.328,0.446,0.638
SRR10142827,0.484,0.328,0.418,0.492,0.448,0.596
SRR10142828,0.486,0.330,0.264,0.328,0.446,0.638
SRR10142829,0.474,0.326,0.402,0.334,0.428,0.618
SRR10142830,0.502,0.318,0.226,0.376,0.428,0.638


,control,Intestinal Diseases,Lung Diseases,IBS,Thyroid Diseases,ColorectalLesions
sample_id,,,,,,
SRR10142826,0.154298,0.010635,0.018288,0.265429,0.293064,0.258285
SRR10142827,0.003539,0.011444,0.229612,0.254421,0.266293,0.234691
SRR10142828,0.006278,0.014139,0.046289,0.314335,0.329002,0.289958
SRR10142829,0.002963,0.011450,0.229745,0.254568,0.266447,0.234826
SRR10142830,0.057314,0.012390,0.015289,0.304597,0.324457,0.285953


# 3. Final SPECTRA evaluation

In [4]:
metadata = pd.read_csv(METADATA_FILE).set_index("sample_id")
metadata.index = metadata.index.astype(str)
metadata = metadata.loc[probability.index]

if set(metadata["true_label"]) != {"IBS", "control"}:
    raise ValueError("Expected IBS and control labels.")

y_binary = metadata["true_label"].eq("IBS").astype(int)
auc = roc_auc_score(y_binary, probability["IBS"])

order = np.argsort(-probability.to_numpy(), axis=1)
ranked = probability.columns.to_numpy()[order]
truth = metadata["true_label"].to_numpy()


auc_table = pd.DataFrame({"AUC": [auc]}, index=["SPECTRA IBS probability"])
display(auc_table)


,AUC
SPECTRA IBS probability,0.857143
